# 03 - Data Quality Pass (Day 8 buffer)

Confirms the cleaned, romanised corpus is trustworthy before any modelling. Checks:
duplicates (within + across sources), label distribution, sanity, and a random spot-check.

### Setup

In [4]:
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Not on Colab / already mounted:', e)

Not on Colab / already mounted: No module named 'google.colab'


In [5]:
import sys; sys.path.insert(0, '/content/drive/MyDrive/dissertation/notebooks')
from hinglish_hate import (load_bohra, load_hasoc2021, load_hasoc2022_threads,
                           build_corpus, filter_romanised)
import pandas as pd
from pathlib import Path
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / 'data'
def find_one(root,name):
    h=list(Path(root).rglob(name));
    if not h: raise FileNotFoundError(name)
    return h[0]
bohra = load_bohra(find_one(DATA_ROOT,'hate_speech.tsv'))
h21_lab=list(DATA_ROOT.rglob('labels.json'))
h21 = load_hasoc2021(h21_lab[0].parents[2]) if h21_lab else None
h22 = load_hasoc2022_threads(DATA_ROOT)
corpus = build_corpus([f for f in [bohra,h21,h22] if f is not None])
roman  = filter_romanised(corpus, include_mixed=True)
print('corpus', len(corpus), '| romanised', len(roman))

[load_bohra] 4574 rows | fixed 0 label typo(s) | dropped 4 unparseable
corpus 11548 | romanised 10694


## 1. Duplicates

`build_corpus` already drops exact `(source, text)` duplicates. Two things it does NOT catch and
that matter here: **within-source case/space near-duplicates**, and **cross-source duplicates**
(the same tweet in two corpora), which would leak between a training corpus and a cross-dataset
test corpus.

In [6]:
norm = corpus['text'].str.lower().str.strip()

# within-source near-dups (case/space-insensitive)
within = corpus[norm.groupby([corpus['source'], norm]).transform('size') > 1]
print('within-source near-duplicate rows:', len(within))

# cross-source exact-text duplicates
dup_text = norm[norm.duplicated(keep=False)]
cross = corpus.loc[dup_text.index].assign(norm=norm.loc[dup_text.index])
cross = cross[cross.groupby('norm')['source'].transform('nunique') > 1]
print('rows whose text also appears in another corpus:', len(cross))
if len(cross):
    display(cross.sort_values('norm')[['source','label','text']].head(20))

within-source near-duplicate rows: 8
rows whose text also appears in another corpus: 0


If cross-source duplicates exist, decide a rule (e.g. keep in the primary/training corpus, drop
from the cross-dataset test set) so the generalisation numbers are not inflated by leakage.

## 2. Label distribution looks sensible

Per-source hate rate should match the published figures (Bohra ~0.36, HASOC21 ~0.46, HASOC22 ~0.51).
A big deviation means a loader or label-mapping problem.

In [7]:
dist = corpus.groupby('source')['label'].agg(['size','mean']).rename(columns={'mean':'hate_rate'})
dist['hate_rate'] = dist['hate_rate'].round(3)
expected = {'bohra2018':0.36, 'hasoc2021':0.46, 'hasoc2022':0.51}
dist['expected'] = dist.index.map(expected)
dist['delta'] = (dist['hate_rate'] - dist['expected']).round(3)
display(dist)
assert corpus['label'].isin([0,1]).all(), 'labels must be binary 0/1'
print('label values OK (all in {0,1})')

,size,hate_rate,expected,delta
source,,,,
bohra2018,4574,0.363,0.36,0.003
hasoc2021,2085,0.495,0.46,0.035
hasoc2022,4889,0.514,0.51,0.004


label values OK (all in {0,1})


## 3. Sanity checks

In [8]:
issues = {}
issues['empty_text']      = int((corpus['text'].str.strip().str.len()==0).sum())
issues['very_short(<3ch)'] = int((corpus['text'].str.len()<3).sum())
issues['null_text']       = int(corpus['text'].isna().sum())
issues['label_not_binary']= int((~corpus['label'].isin([0,1])).sum())
# every romanised row should be mostly_latin or mixed_script, never mostly_devanagari
issues['devanagari_in_roman'] = int((roman['script']=='mostly_devanagari').sum())
print(pd.Series(issues, name='count'))
print('\nscript breakdown (full corpus):', dict(corpus['script'].value_counts()))

empty_text             0
very_short(<3ch)       0
null_text              0
label_not_binary       0
devanagari_in_roman    0
Name: count, dtype: int64

script breakdown (full corpus): {'mostly_latin': np.int64(10242), 'mostly_devanagari': np.int64(854), 'mixed_script': np.int64(452)}


## 4. Random spot-check

Read a random 15 rows with their source/label/script together - the quickest way to catch a
mislabelled or misparsed row.

In [9]:
corpus.sample(15, random_state=7)[['source','label','script','text']]

,source,label,script,text
7432,hasoc2022,1,mostly_latin,"@NooriKhanINC @khanumarfa For you Twitter, Fac..."
10914,hasoc2022,1,mostly_latin,@SameerGojri @RanaAyyub Madarshachaap spotted 😂
5863,hasoc2021,1,mostly_latin,@SaffronSunanda @grrajan3 💯%
7208,hasoc2022,0,mostly_latin,@__srkian555__ @SRKian_Arijit @iamsrk SRKians ...
8209,hasoc2022,0,mostly_latin,@_sabanaqvi @derekobrienmp You must have been ...
3284,bohra2018,0,mostly_latin,Janab 4 shadiya karna rape bhi
2369,bohra2018,1,mostly_latin,I hate vikas gupti plz shilpa ka naam is Banda...
4444,bohra2018,1,mostly_latin,Hate this gwar sadakchhap TUCHCHA
7001,hasoc2022,0,mostly_latin,@AmitkymarV @_sabanaqvi Her take is that she h...
8354,hasoc2022,1,mostly_latin,@FakharZamanLive New Zea Land Just Killed Paki...


## 5. Fixes + re-verify

Apply any fix decided above (example below drops cross-source duplicate rows from the
*non-primary* corpus), then re-run cells 1-4 to confirm the issue is gone. Leave commented if
no fix is needed.

In [10]:
# Example fix - drop cross-source dup rows from HASOC (keep Bohra as primary):
# keep = ~((corpus['source']!='bohra2018') & norm.isin(
#           norm[corpus['source']=='bohra2018']))
# corpus = corpus[keep].reset_index(drop=True)
# print('after fix:', len(corpus))
print('No fix applied. If clean, this corpus is trustworthy for modelling.')

No fix applied. If clean, this corpus is trustworthy for modelling.
